# Edge IIoT - Binary Classification


In [1]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
import torch

print(torch.__version__) # X.XX.X+cuXXX / If X.XX.X+cpu it won't work
print(torch.cuda.is_available()) # False
print(torch.version.cuda) # None or mismatched version
print(torch.cuda.device_count()) # 0

2.5.1+cu121
True
12.1
1


In [3]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from src.config import DATASETS, SEED
dataset_name = "edge_iiot"
config = DATASETS[dataset_name]

print("\n--- Dataset Information ---")
print(f"Name: {dataset_name.upper()}")
print(f"Path: {config['processed_path']}\n")

filename = "ML-EdgeIIoT-dataset"

csv_path = config['processed_path'] / f"{filename}.pkl"
print(f"CSV Path: {csv_path}\n")

print("Loading dataset... (This may take a while)")
df = pd.read_pickle(csv_path)

print(f"\nDataset loaded with shape: {df.shape}")


--- Dataset Information ---
Name: EDGE_IIOT
Path: /home/uo294319/ML-NIDS-IIoT/data/edge_iiot/processed

CSV Path: /home/uo294319/ML-NIDS-IIoT/data/edge_iiot/processed/ML-EdgeIIoT-dataset.pkl

Loading dataset... (This may take a while)

Dataset loaded with shape: (152590, 62)


## 1. Undersampling and Balancing

In [4]:
target     = 'Attack_label'
target_str = 'Attack_type'

In [5]:
y_str = df[target_str]
df_no_label = df.drop(columns=[target_str])

In [6]:
MAX_PRESENCE = 0.02

counts = y_str.value_counts()
total_rows = len(y_str)

print(counts)
print("\nTOTAL: ", total_rows)

Attack_type
Normal                   24301
DDoS_UDP                 14498
DDoS_ICMP                13307
Ransomware               10925
DDoS_HTTP                10561
SQL_injection            10311
Uploading                10269
Backdoor                 10195
Vulnerability_scanner    10075
Port_Scanning            10071
XSS                      10051
Password                  9989
DDoS_TCP                  6011
MITM                      1028
Fingerprinting             998
Name: count, dtype: int64

TOTAL:  152590


In [7]:
print(f"{'Attack Type':<25} | {'Old Count':<15} | {'New Count':<15}\n" + "-"*55)

sampling_strategy = {}
for attack_type, count in counts.items():
    current_presence = count / total_rows

    if current_presence > MAX_PRESENCE:
        new_count = int(total_rows * MAX_PRESENCE)
    else:
        new_count = count

    sampling_strategy[attack_type] = new_count
    print(f"{attack_type:<25} | {count:<15} | {new_count:<15}")


print("-"*55 + f"\n{'TOTAL':<25} | {counts.sum():<15} | {sum(sampling_strategy.values()):<15}")


Attack Type               | Old Count       | New Count      
-------------------------------------------------------
Normal                    | 24301           | 3051           
DDoS_UDP                  | 14498           | 3051           
DDoS_ICMP                 | 13307           | 3051           
Ransomware                | 10925           | 3051           
DDoS_HTTP                 | 10561           | 3051           
SQL_injection             | 10311           | 3051           
Uploading                 | 10269           | 3051           
Backdoor                  | 10195           | 3051           
Vulnerability_scanner     | 10075           | 3051           
Port_Scanning             | 10071           | 3051           
XSS                       | 10051           | 3051           
Password                  | 9989            | 3051           
DDoS_TCP                  | 6011            | 3051           
MITM                      | 1028            | 1028           
Fingerprinting

In [8]:
from imblearn.under_sampling import RandomUnderSampler

rus = RandomUnderSampler(sampling_strategy=sampling_strategy, random_state=SEED)
df_no_label, y_str = rus.fit_resample(df_no_label, y_str)

print(df_no_label.shape)

(41689, 61)


In [9]:
# X/y split
X     = df_no_label.drop(columns=[target])
y     = df_no_label[target]

## 1. Pre-processing

In [10]:
# Select only numeric
print("--- Non-numeric cols to drop ---\n\n", X.select_dtypes(include=['str', 'object', 'category']).columns)

X = X.select_dtypes(include=['number'])

print("\n\nRemaining categorical cols:", len(X.select_dtypes(include=['str', 'object', 'category']).columns))

--- Non-numeric cols to drop ---

 Index(['http.file_data', 'http.referer', 'http.request.path',
       'http.request.uri.query', 'http.request.version', 'mqtt.msg', 'proto'],
      dtype='str')


Remaining categorical cols: 0


In [11]:
# Remove env-specific columns
cols_to_drop = [
    'frame.time.delta', 'frame.time.order',
    *[c for c in X.columns if c.startswith('ip.src_category') or c.startswith('ip.dst_category')]
]
X = X.drop(columns=cols_to_drop, errors='ignore')

In [12]:
print(f"NaN values in target variable: {y.isna().sum()}")
print(f"NaN values in features: {X.isna().sum().sum()}")

NaN values in target variable: 0
NaN values in features: 0


In [13]:
X_train, X_test, y_train, y_test, y_str_train, y_str_test = train_test_split(
    X, y, y_str, test_size=0.2, random_state=SEED, stratify=y_str
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

X_train shape: (33351, 43)
X_test shape: (8338, 43)


In [14]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model_results = {}

## 2. LazyPredict
[Docs](https://pypi.org/project/lazypredict/)

In [15]:
from lazypredict.Supervised import LazyClassifier

# With categorical encoding, timeout, cross-validation, and GPU
clf = LazyClassifier(
    verbose=1,                          # Show progress
    ignore_warnings=True,               # Suppress warnings
    custom_metric=None,                 # Use default metrics
    predictions=False,                  # Don't Return predictions
    classifiers='all',                  # Use all available classifiers
    timeout=60,                         # Max time per model in seconds
    cv=5,                               # Cross-validation folds (optional)
)

models, _ = clf.fit(X_train, X_test, y_train, y_test)
print("\n--- Models Evaluated ---")

  0%|          | 0/32 [00:00<?, ?it/s]

/home/uo294319/ML-NIDS-IIoT/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:71: FutureWarning: Class PassiveAggressiveClassifier is deprecated; this is deprecated in version 1.8 and will be removed in 1.10. Use `SGDClassifier(loss='hinge', penalty=None, learning_rate='pa1', eta0=1.0)` instead.
  warnings.warn(msg, category=FutureWarning)
/home/uo294319/ML-NIDS-IIoT/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:71: FutureWarning: Class PassiveAggressiveClassifier is deprecated; this is deprecated in version 1.8 and will be removed in 1.10. Use `SGDClassifier(loss='hinge', penalty=None, learning_rate='pa1', eta0=1.0)` instead.
  warnings.warn(msg, category=FutureWarning)
/home/uo294319/ML-NIDS-IIoT/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:71: FutureWarning: Class PassiveAggressiveClassifier is deprecated; this is deprecated in version 1.8 and will be removed in 1.10. Use `SGDClassifier(loss='hinge', penalty=None, learning_rate='pa1


--- Models Evaluated ---


In [16]:
display(models)

,Accuracy,Balanced Accuracy,ROC AUC,F1 Score,Precision,Recall,Accuracy CV Mean,Accuracy CV Std,Balanced Accuracy CV Mean,Balanced Accuracy CV Std,ROC AUC CV Mean,ROC AUC CV Std,F1 Score CV Mean,F1 Score CV Std,Precision CV Mean,Precision CV Std,Recall CV Mean,Recall CV Std,Time Taken
Model,,,,,,,,,,,,,,,,,,,
LGBMClassifier,0.988726,0.923706,0.992449,0.988300,0.988836,0.988726,0.988816,0.000507,0.923975,0.003800,0.992561,0.000638,0.988391,0.000550,0.988938,0.000485,0.988816,0.000507,349.319227
XGBClassifier,0.988367,0.923512,0.992349,0.987942,0.988412,0.988367,0.988486,0.000737,0.922854,0.004266,0.992129,0.000654,0.988049,0.000783,0.988576,0.000747,0.988486,0.000737,2.308334
KNeighborsClassifier,0.988367,0.920492,0.970948,0.987900,0.988511,0.988367,0.988246,0.000573,0.920271,0.003646,0.972336,0.008649,0.987777,0.000616,0.988373,0.000572,0.988246,0.000573,1.292729
ExtraTreesClassifier,0.986328,0.906557,0.991851,0.985673,0.986526,0.986328,0.985728,0.001618,0.902689,0.011004,0.992248,0.000618,0.985003,0.001782,0.985938,0.001574,0.985728,0.001618,1.169164
BaggingClassifier,0.986328,0.906557,0.985163,0.985673,0.986526,0.986328,0.984948,0.000762,0.897362,0.004896,0.986113,0.002313,0.984147,0.000843,0.985179,0.000755,0.984948,0.000762,1.711095
RandomForestClassifier,0.986328,0.906557,0.986806,0.985673,0.986526,0.986328,0.985008,0.000716,0.897582,0.004902,0.987565,0.001349,0.984210,0.000798,0.985247,0.000693,0.985008,0.000716,2.613703
DecisionTreeClassifier,0.986328,0.906557,0.985194,0.985673,0.986526,0.986328,0.984948,0.000675,0.897551,0.004786,0.984731,0.001619,0.984151,0.000754,0.985172,0.000649,0.984948,0.000675,1.490926
ExtraTreeClassifier,0.986208,0.906493,0.991480,0.985554,0.986377,0.986208,0.986957,0.001360,0.912035,0.009736,0.989850,0.001681,0.986374,0.001499,0.987098,0.001308,0.986957,0.001360,1.265575
AdaBoostClassifier,0.981290,0.873641,0.970973,0.980050,0.981567,0.981290,0.978951,0.002843,0.859977,0.019771,0.968400,0.003755,0.977368,0.003356,0.979170,0.002743,0.978951,0.002843,2.246882


In [17]:
display(models.sort_values(by='F1 Score CV Mean', ascending=False).head(5))

,Accuracy,Balanced Accuracy,ROC AUC,F1 Score,Precision,Recall,Accuracy CV Mean,Accuracy CV Std,Balanced Accuracy CV Mean,Balanced Accuracy CV Std,ROC AUC CV Mean,ROC AUC CV Std,F1 Score CV Mean,F1 Score CV Std,Precision CV Mean,Precision CV Std,Recall CV Mean,Recall CV Std,Time Taken
Model,,,,,,,,,,,,,,,,,,,
LGBMClassifier,0.988726,0.923706,0.992449,0.988300,0.988836,0.988726,0.988816,0.000507,0.923975,0.003800,0.992561,0.000638,0.988391,0.000550,0.988938,0.000485,0.988816,0.000507,349.319227
XGBClassifier,0.988367,0.923512,0.992349,0.987942,0.988412,0.988367,0.988486,0.000737,0.922854,0.004266,0.992129,0.000654,0.988049,0.000783,0.988576,0.000747,0.988486,0.000737,2.308334
KNeighborsClassifier,0.988367,0.920492,0.970948,0.987900,0.988511,0.988367,0.988246,0.000573,0.920271,0.003646,0.972336,0.008649,0.987777,0.000616,0.988373,0.000572,0.988246,0.000573,1.292729
ExtraTreeClassifier,0.986208,0.906493,0.991480,0.985554,0.986377,0.986208,0.986957,0.001360,0.912035,0.009736,0.989850,0.001681,0.986374,0.001499,0.987098,0.001308,0.986957,0.001360,1.265575
ExtraTreesClassifier,0.986328,0.906557,0.991851,0.985673,0.986526,0.986328,0.985728,0.001618,0.902689,0.011004,0.992248,0.000618,0.985003,0.001782,0.985938,0.001574,0.985728,0.001618,1.169164
